<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# C18's Threshold Hunch, Run Again

The [loose ends](c18_loose_ends.ipynb) swept the activation threshold of the conserved budget at c = 0.6 (a gain of
1.2) and came out void: its control, L0, missed by 0.001, largely because D was located on a grid of step 0.125 and
compared with the run's 1.528, located on a grid of step 0.25. This notebook runs the same sweep again with fresh
seeds, as registered in the journal of 2026-11-05, § C18's Threshold Hunch, Run Again: the Plan. The model, the graph,
the dynamics, the replacement of a seed with no out-edge and the 1% locator are the run's, copied unchanged from
`c18_loose_ends.ipynb`. What changes: L0 locates D on the step-0.25 points of this sweep, the run's own grid, and its
tolerance comes from the spread of two independent 50-draw estimates, computed from draws already seen; and the fall
of D is read with a bootstrap interval, its direction fixed.


In [1]:
import multiprocessing as mp
import os
import warnings

import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*fork.*")

ROOT = 2026110518  # fresh: no notebook, crate or journal entry used it before this run
OLD = 20261103  # the run's and the loose ends' root, for the calibration only
WORKERS = 2
N, DRAWS, BUDGET = 10000, 50, 0.6
GRID = np.arange(0.5, 3.0001, 0.125)
STEP_025 = GRID[::2]  # 0.5, 0.75, ..., 3.0: the run's own grid points in this range
THRESHOLDS = [0.1, 0.03, 0.01, 0.003, 0.001]
REACH_LEVEL = 0.01
SETTLE_TOL, SETTLE_STEPS = 1e-10, 20000
DECAY = 0.5
RUN_D = 1.528  # the run's D for the budget at c = 0.6 (c18_cliff_run.ipynb, cell 7), on its grid of step 0.25
TOL_REGISTERED = 0.0583  # L0's tolerance as computed before the run (the journal's plan); recomputed below
CAL_BOOT, DELTA_BOOT, LEVEL = 20000, 2000, 0.99


def network(n, d, rng):
    """The pilot's directed random graph: Poisson(d n) edges with uniform ends, self-loops dropped, repeats kept."""
    m = rng.poisson(d * n)
    src, dst = rng.integers(0, n, m), rng.integers(0, n, m)
    keep = src != dst
    return src[keep], dst[keep]


def out_component(n, src, dst, seed=0):
    """A mask of the units the seed reaches along directed edges, the seed included."""
    reached = np.zeros(n, dtype=bool)
    reached[seed] = True
    frontier = np.array([seed])
    while frontier.size:
        on = np.zeros(n, dtype=bool)
        on[frontier] = True
        new = dst[on[src]]
        new = np.unique(new[~reached[new]])
        reached[new] = True
        frontier = new
    return reached


def steady(n, src, dst, budget):
    """The run's dynamics with the budget: the seed held at 1; each step a unit becomes half its last value plus its
    input, each unit's output (budget x activation) divided equally among its targets, clipped to [0, 1]; until no
    unit moves by more than 1e-10 or for 20 000 steps. Returns the activations and whether they settled."""
    outdeg = np.bincount(src, minlength=n)
    w = budget / outdeg[src]
    a = np.zeros(n)
    a[0] = 1.0
    for _ in range(SETTLE_STEPS):
        drive = np.bincount(dst, weights=w * a[src], minlength=n)
        new = np.clip(DECAY * a + drive, 0.0, 1.0)
        new[0] = 1.0
        moved = np.max(np.abs(new - a))
        a = new
        if moved <= SETTLE_TOL:
            return a, True
    return a, False


def eligible(d, entropy):
    """A graph at mean degree d whose seed has an out-edge; a replaced graph is the next from the same generator."""
    rng = np.random.default_rng(entropy)
    replaced = 0
    while True:
        src, dst = network(N, d, rng)
        if np.any(src == 0):
            return src, dst, replaced
        replaced += 1


def calibration_draw(task):
    """One already-seen draw at threshold 0.1, by the seeds of the run or of the loose ends."""
    d, entropy = task
    src, dst, _ = eligible(d, entropy)
    a, settled = steady(N, src, dst, BUDGET)
    return float(np.mean(a[1:] > 0.1)), settled


def sweep_draw(task):
    """One fresh draw at GRID[point]: the fraction of the other units above each threshold, the out-component's
    fraction, whether it settled, and how many graphs were replaced."""
    point, i = task
    src, dst, replaced = eligible(float(GRID[point]), [ROOT, 1, point, i])
    a, settled = steady(N, src, dst, BUDGET)
    above = [float(np.mean(a[1:] > t)) for t in THRESHOLDS]
    topo = float((out_component(N, src, dst).sum() - 1) / (N - 1))
    return above, topo, settled, replaced


def locate_on(grid, means):
    """The run's rule: D is where the mean reached fraction first passes 1%, interpolated linearly from the grid
    point before; NaN if it never does."""
    passing = means >= REACH_LEVEL
    if not passing.any():
        return np.nan
    i = int(np.argmax(passing))
    if i == 0:
        return float(grid[0])
    return float(grid[i - 1] + (REACH_LEVEL - means[i - 1]) * (grid[i] - grid[i - 1]) / (means[i] - means[i - 1]))


def pool_map(fn, tasks):
    with mp.get_context("fork").Pool(WORKERS) as pool:
        return pool.map(fn, tasks, chunksize=10)


print("numpy", np.__version__, f"- fresh generator seeds [{ROOT}, 1, point, draw]; {WORKERS} worker processes")


numpy 2.5.4 - fresh generator seeds [2026110518, 1, point, draw]; 2 worker processes


## L0's tolerance, from draws already seen

The run's D, 1.528, is one estimate from 50 draws a point. A fresh estimate from 50 other draws differs from it by
chance, and the locator is far from linear in the mean at d = 1.75, so the spread is taken from the draws, not from a
formula. The run's 50 draws (seeds `[20261103, 1, 6, point, 10000, draw]`) and the loose ends' 50 (seeds
`[20261103, 11, point, draw]`) at threshold 0.1 are drawn again at d = 1.0 to 2.0 by 0.25. That they reproduce the
run's 1.528 and the loose ends' 1.569 on the step-0.25 grid checks the copied harness. The 100 draws a point are then
pooled; two independent sets of 50 draws a point are resampled from them, 20 000 times each, D located on each, and
the tolerance is the 99th percentile of the absolute difference.


In [2]:
CAL_GRID = np.arange(1.0, 2.0001, 0.25)
run_points = [int(round((d - 0.25) / 0.25)) for d in CAL_GRID]  # indices on the run's grid 0.25 to 12
le_points = [int(round((d - 0.5) / 0.125)) for d in CAL_GRID]  # indices on the loose ends' grid 0.5 to 3
tasks = [(float(d), [OLD, 1, 6, p, N, i]) for d, p in zip(CAL_GRID, run_points) for i in range(DRAWS)]
tasks += [(float(d), [OLD, 11, p, i]) for d, p in zip(CAL_GRID, le_points) for i in range(DRAWS)]
out = pool_map(calibration_draw, tasks)
assert all(s for _, s in out)
cal = np.array([v for v, _ in out]).reshape(2, CAL_GRID.size, DRAWS)
D_run, D_le = (locate_on(CAL_GRID, cal[k].mean(axis=1)) for k in range(2))
print(f"the run's draws: D = {D_run:.3f}; the loose ends' draws on the step-0.25 grid: D = {D_le:.3f}")
assert round(D_run, 3) == RUN_D and round(D_le, 3) == 1.569
pooled = np.concatenate([cal[0], cal[1]], axis=1)
rng = np.random.default_rng([ROOT, 0])


def resampled_d():
    idx = rng.integers(0, pooled.shape[1], (CAL_BOOT, CAL_GRID.size, DRAWS))
    means = pooled[np.arange(CAL_GRID.size)[None, :, None], idx].mean(axis=2)
    return np.array([locate_on(CAL_GRID, m) for m in means])


first, second = resampled_d(), resampled_d()
TOL = float(np.quantile(np.abs(first - second), LEVEL))
print("pooled means by d: " + ", ".join(f"{d:.2f}: {m:.4f}" for d, m in zip(CAL_GRID, pooled.mean(axis=1))))
print(f"L0's tolerance, the 99th percentile of |D - D'| over {CAL_BOOT} pairs: {TOL:.4f}")
assert round(TOL, 4) == TOL_REGISTERED


the run's draws: D = 1.528; the loose ends' draws on the step-0.25 grid: D = 1.569


pooled means by d: 1.00: 0.0009, 1.25: 0.0018, 1.50: 0.0021, 1.75: 0.0513, 2.00: 0.5263
L0's tolerance, the 99th percentile of |D - D'| over 20000 pairs: 0.0583


## The sweep, with fresh seeds

Every draw at every grid point, read at every threshold; the graph's own reach (the seed's out-component) off the
same draws.


In [3]:
results = pool_map(sweep_draw, [(p, i) for p in range(GRID.size) for i in range(DRAWS)])
above = np.array([r[0] for r in results]).reshape(GRID.size, DRAWS, len(THRESHOLDS))
topo = np.array([r[1] for r in results]).reshape(GRID.size, DRAWS)
unsettled = sum(not r[2] for r in results)
replaced = sum(r[3] for r in results)
means = above.mean(axis=1)
D = {t: locate_on(GRID, means[:, j]) for j, t in enumerate(THRESHOLDS)}
D25 = {t: locate_on(STEP_025, means[::2, j]) for j, t in enumerate(THRESHOLDS)}
D_topo, D_topo25 = locate_on(GRID, topo.mean(axis=1)), locate_on(STEP_025, topo.mean(axis=1)[::2])
print(f"{GRID.size} grid points x {DRAWS} draws at N = {N}; unsettled {unsettled}; graphs replaced {replaced}")
print("threshold      D (step 0.125)   D (step 0.25)")
for t in THRESHOLDS:
    print(f"{t:<13}  {D[t]:.3f}            {D25[t]:.3f}")
print(f"out-component  {D_topo:.3f}            {D_topo25:.3f}")
print("\nmean reached fraction by d (columns: thresholds, then the out-component)")
for k, d in enumerate(GRID):
    print(f"{d:5.3f}  " + "  ".join(f"{m:.4f}" for m in means[k]) + f"  {topo[k].mean():.4f}")


21 grid points x 50 draws at N = 10000; unsettled 1; graphs replaced 353
threshold      D (step 0.125)   D (step 0.25)
0.1            1.639            1.539
0.03           1.507            1.501
0.01           1.328            1.291
0.003          1.210            1.211
0.001          1.128            1.118
out-component  1.010            1.011

mean reached fraction by d (columns: thresholds, then the out-component)
0.500  0.0003  0.0003  0.0003  0.0003  0.0003  0.0003
0.625  0.0003  0.0003  0.0003  0.0003  0.0003  0.0003
0.750  0.0005  0.0005  0.0005  0.0005  0.0005  0.0005
0.875  0.0008  0.0010  0.0011  0.0011  0.0012  0.0013
1.000  0.0008  0.0010  0.0012  0.0014  0.0016  0.0026
1.125  0.0013  0.0025  0.0040  0.0065  0.0098  0.0922
1.250  0.0016  0.0033  0.0061  0.0116  0.0194  0.1688
1.375  0.0018  0.0052  0.0123  0.0306  0.0613  0.3378
1.500  0.0025  0.0093  0.0295  0.0849  0.1669  0.4673
1.625  0.0049  0.0218  0.0700  0.1773  0.2922  0.4837
1.750  0.0500  0.1494  0.2849  0.4281  

## The registered reading

The fall, Delta = D(0.1) - D(0.001) on the step-0.125 grid, with a percentile bootstrap interval at 0.99: draws
resampled with replacement within each grid point, the same resample read at both thresholds, 2000 times.


In [4]:
rng = np.random.default_rng([ROOT, 2])
idx = rng.integers(0, DRAWS, (DELTA_BOOT, GRID.size, DRAWS))
ends = [0, len(THRESHOLDS) - 1]
boot_means = above[np.arange(GRID.size)[None, :, None], idx][..., ends].mean(axis=2)  # (resample, grid, 2)
boot_delta = np.array([locate_on(GRID, m[:, 0]) - locate_on(GRID, m[:, 1]) for m in boot_means])
delta = D[0.1] - D[0.001]
tail = (1 - LEVEL) / 2
lo, hi = (float(np.quantile(boot_delta, q)) for q in (tail, 1 - tail)) if np.all(np.isfinite(boot_delta)) else (np.nan, np.nan)
print(f"Delta = D(0.1) - D(0.001) = {delta:.3f}, interval at {LEVEL} [{lo:.3f}, {hi:.3f}];"
      f" resamples with no D: {int(np.sum(~np.isfinite(boot_delta)))}")

# The registered rules (journal of 2026-11-05, § C18's Threshold Hunch, Run Again: the Plan).
L0 = abs(D25[0.1] - RUN_D) <= TOL
rise = any(not (D[b] <= D[a] + 0.125) for a, b in zip(THRESHOLDS, THRESHOLDS[1:]))
L1 = not rise and lo > 0
L2 = abs(D[0.001] - D_topo) <= 0.25
refuted = not (D[0.001] < 1.4) or rise or not (hi > 0)
print(f"L0, the run's D reproduced on the step-0.25 grid (|{D25[0.1]:.3f} - {RUN_D}| <= {TOL:.4f}): {L0}")
print(f"L1, D falls: no step rises by more than 0.125 ({not rise}), and Delta's interval lies above 0 ({lo > 0}): {L1}")
print(f"L2, D at 0.001 within 0.25 of the out-component's D ({D[0.001]:.3f} against {D_topo:.3f}): {L2}")
print(f"refutation, D at 0.001 at or above 1.4, a rise, or Delta's interval wholly at or below 0: {refuted}")
print("reading:", "void (L0 fails)" if not L0 else "hunch refuted" if refuted else "hunch supported" if L1 and L2 else
      "partly: D does not fall clearly, or falls but does not reach the graph's own")


Delta = D(0.1) - D(0.001) = 0.511, interval at 0.99 [0.427, 0.568]; resamples with no D: 0
L0, the run's D reproduced on the step-0.25 grid (|1.539 - 1.528| <= 0.0583): True
L1, D falls: no step rises by more than 0.125 (True), and Delta's interval lies above 0 (True): True
L2, D at 0.001 within 0.25 of the out-component's D (1.128 against 1.010): True
refutation, D at 0.001 at or above 1.4, a rise, or Delta's interval wholly at or below 0: False
reading: hunch supported
